# 2 lineage

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
============================================================
Case1 BMMC 双谱系动态 GRN — 最终主图脚本 (a–g)  [双谱系版]
============================================================
本案例聚焦髓系 (myeloid) 与红系 (erythroid) 两条分化轨迹。
  · 特异性 = 两谱系最大活性占比, 取值 [1/2, 1], 无特异性基线 = 0.5
  · 度量统一为 n_targets
  · 无偏发现 → 金标准超几何验证 (基线 N/K 在双谱系池内重算)
  · 预测=主体, 验证=★+加粗
============================================================
"""
import os, pickle
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats
from scipy.stats import hypergeom
from scipy.ndimage import uniform_filter1d

# ============================================================
# 0. 参数 —— 仅两条谱系
# ============================================================
data_root  = "/home/wuyan/dygmamba_project/NewRealPlan/Case1/process/"
output_dir = "/home/wuyan/dygmamba_project/DRIMA/data/case1/" + "analysis_final_2lineage/"
os.makedirs(output_dir, exist_ok=True)

N_TIME_BINS  = 15
VALUE_COL    = 'n_targets'
TOP_N_DISC   = 30
MIN_ACTIVITY = 20
C_MAX_ROWS   = {'myeloid': 12, 'erythroid': 12}
G_N_COL      = {'myeloid': 8,  'erythroid': 8}
# e 图 x 轴范围：双谱系 specificity ∈ [0.5,1]，跑后按实际微调
E_XLIM       = (0.48, 1.00)

GOLDSTANDARD = {
    "myeloid":   ["SPI1","CEBPA","CEBPB","IRF8","KLF4","EGR1","MAFB","MAF","NR4A1","JUNB","FOS","IRF4"],
    "erythroid": ["GATA1","GATA2","KLF1","TAL1","LMO2","LDB1","NFE2","GFI1B","TCF3","ZBTB7A"],
}
trajectories = {
    "myeloid":   {"path": data_root+"myeloid/process/",
                  "cell_order": ["HSC","G/M prog","CD14+ Mono"],
                  "color": "#E41A1C", "known_tfs": GOLDSTANDARD["myeloid"]},
    "erythroid": {"path": data_root+"erythroid/process/",
                  "cell_order": ["HSC","MK/E prog","Erythroblast"],
                  "color": "#FF7F00", "known_tfs": GOLDSTANDARD["erythroid"]},
}
lineage_order = list(trajectories.keys())       # ['myeloid','erythroid']
L = len(lineage_order)                           # = 2
BASELINE = 1.0 / L                               # = 0.5
time_axis = np.linspace(0, 1, N_TIME_BINS)

# ============================================================
# helper
# ============================================================
def split_motif(name): return [p.strip() for p in str(name).split('::')]
def is_validated(tf, gold): return bool(set(split_motif(tf)) & gold)
def get_gold(traj): return set(trajectories[traj]['known_tfs'])
def pstr(p): return "n.s." if (p is None or p > 0.05) else f"p={p:.1e}"
def short(tf, m=16): return tf if len(tf) <= m else tf[:m-1]+'…'
def add_label(ax, lab):
    ax.text(-0.08, 1.06, lab, transform=ax.transAxes, fontsize=16,
            fontweight='bold', va='top')

# ============================================================
# 1. 加载 (仅两条)
# ============================================================
print("="*60 + "\nStep 1: 加载双谱系 GRN\n" + "="*60)
all_data = {}
for traj, cfg in trajectories.items():
    p = cfg['path']
    all_data[traj] = {
        'adata':       ad.read_h5ad(p+"rna_processed.h5ad"),
        'tf_gene_dyn': pd.read_pickle(p+"pred_time_tf_gene.pkl"),
        'tf_gene_avg': pd.read_pickle(p+"pred_tf_gene.pkl"),
        'region_gene': pd.read_pickle(p+"pred_peak_gene.pkl"),
        'tf_region':   pd.read_pickle(p+"pred_tf_peak.pkl"),
        'node_id':     pd.read_pickle(p+"node_id.pkl"),
        'cfg': cfg,
    }
    print(f"  {traj}: dyn={len(all_data[traj]['tf_gene_dyn'])}, "
          f"cells={all_data[traj]['adata'].shape[0]}")

# ============================================================
# 2. 计算函数
# ============================================================
def normalize_series(values, method):
    v = np.array(values, float)
    if method == 'zscore':
        s = v.std(); return (v-v.mean())/s if s>1e-8 else np.zeros_like(v)
    if method == 'minmax':
        r = v.max()-v.min(); return (v-v.min())/r if r>1e-8 else np.zeros_like(v)
    if method == 'maxnorm':
        return v/v.max() if v.max()>1e-8 else np.zeros_like(v)
    return v

def compute_tf_activity_curves(dyn, n_bins=N_TIME_BINS):
    df = dyn.copy()
    df['time_bin'] = pd.cut(df['ts'], bins=n_bins, labels=False)
    act = df.groupby(['TF','time_bin']).agg(
        n_targets=('Gene','nunique'),
        mean_weight=('avg_weight','mean'),
        total_weight=('avg_weight','sum')).reset_index()
    act['activity_raw'] = act['n_targets']
    for m in ['zscore','minmax','maxnorm']:
        act[f'activity_{m}'] = act.groupby('TF')['activity_raw'].transform(
            lambda x: normalize_series(x.values, m))
    act['activity'] = act['activity_raw']
    return act

def get_tf_curve(act, tf, n_bins=N_TIME_BINS, col='activity_raw'):
    sub = act[act['TF']==tf]; curve = np.zeros(n_bins)
    for _, r in sub.iterrows():
        b = int(r['time_bin'])
        if 0 <= b < n_bins: curve[b] = r[col]
    return curve

def safe_curve(act, tf, col):
    avail = set(act['TF'].values)
    q = tf if tf in avail else next((s for s in split_motif(tf) if s in avail), None)
    return None if q is None else np.asarray(get_tf_curve(act, q, col=col), float)

def compute_tf_outdegree(avg):
    return avg.groupby('TF').agg(
        n_targets=('Gene','nunique'),
        mean_weight=('avg_ts_weight','mean'),
        sum_weight=('avg_ts_weight','sum')).reset_index()

def compute_grn_turnover(dyn, n_bins=N_TIME_BINS, thr=0.5, smooth=3):
    d = dyn.copy(); d['time_bin'] = pd.cut(d['ts'], bins=n_bins, labels=False)
    a = d[d['avg_weight']>thr]
    bs = {b: set(zip(a[a['time_bin']==b]['TF'], a[a['time_bin']==b]['Gene']))
          for b in range(n_bins)}
    t = np.zeros(n_bins)
    for b in range(1, n_bins):
        u = bs[b-1]|bs[b]; i = bs[b-1]&bs[b]
        t[b] = 1-len(i)/len(u) if u else 0.0
    t[0] = t[1]
    return t, uniform_filter1d(t, size=smooth), np.array([len(bs[b]) for b in range(n_bins)])

for traj, d in all_data.items():
    d['tf_activity'] = compute_tf_activity_curves(d['tf_gene_dyn'])
tf_outdegrees = {}
for traj, d in all_data.items():
    tf_outdegrees[traj] = compute_tf_outdegree(d['tf_gene_avg'])
    tf_outdegrees[traj]['trajectory'] = traj
for traj, d in all_data.items():
    raw, sm, cnt = compute_grn_turnover(d['tf_gene_dyn'])
    d['turnover'] = {'raw': raw, 'smooth': sm, 'edge_counts': cnt}

# ============================================================
# 3. 无偏发现 → 验证 (双谱系: 分母只有两列, 基线 0.5)
# ============================================================
def compute_specificity_matrix(tf_outdegrees, lineage_order, value_col=VALUE_COL):
    all_tfs = set()
    for t in lineage_order: all_tfs.update(tf_outdegrees[t]['TF'].values)
    all_tfs = sorted({t for t in all_tfs if not str(t).startswith('chr')})
    mat = pd.DataFrame(index=all_tfs)
    for t in lineage_order:
        od = tf_outdegrees[t].set_index('TF')
        mat[t] = od[value_col].reindex(mat.index).fillna(0.0)
    mat = mat[mat[lineage_order].sum(axis=1) > 0]
    prop = mat[lineage_order].div(mat[lineage_order].sum(axis=1), axis=0)
    mat['specificity'] = prop.max(axis=1)        # ∈ [0.5, 1] for L=2
    mat['best_lineage'] = prop.idxmax(axis=1)
    return mat, prop

def discover_top_markers(sm, lineage_order, top_n=TOP_N_DISC, min_activity=MIN_ACTIVITY):
    mk = {}
    for ln in lineage_order:
        sub = sm[sm['best_lineage']==ln].copy()
        sub = sub[sub[ln] >= min_activity]
        sub = sub.sort_values('specificity', ascending=False).head(top_n)
        mk[ln] = sub.reset_index().rename(columns={'index':'TF'})
    return mk

def validate_against_known(sm, mk, trajectories, lineage_order, top_n=TOP_N_DISC):
    pool = set()
    for name in sm.index: pool |= set(split_motif(name))
    N = len(pool); results = {}
    print("="*64 + f"\n双谱系 无偏发现 → 金标准验证  N={N}\n" + "="*64)
    for ln in lineage_order:
        known = set(trajectories[ln]['known_tfs']); kip = known & pool; K = len(kip)
        disc = list(mk[ln]['TF']); hit, names = set(), []
        for nm in disc:
            inter = set(split_motif(nm)) & kip
            if inter: hit |= inter; names.append(nm)
        k, n = len(hit), len(disc)
        pval = hypergeom.sf(k-1, N, K, n) if (K>0 and k>0) else 1.0
        exp = K*n/N if N else 0.0; fold = k/exp if exp>0 else 0.0
        results[ln] = {'N':N,'K':K,'n':n,'k':k,'hits':hit,'fold':fold,
                       'hit_marker_names':set(names),'pval':pval,'discovered':disc}
        print(f"[{ln}] K={K}, k={k}/{K}, 富集={fold:.1f}x, p={pval:.2e}, ★={sorted(set(names))}")
    return results

spec_mat, prop = compute_specificity_matrix(tf_outdegrees, lineage_order)
markers    = discover_top_markers(spec_mat, lineage_order)
validation = validate_against_known(spec_mat, markers, trajectories, lineage_order)
for ln, df in markers.items():
    df.to_csv(output_dir + f"discovered_markers_{ln}.csv", index=False)

# 打印实际 specificity 范围，便于核对 e 图 x 轴
print("\nspecificity 实际范围 (用于 e 图 x 轴):")
for ln in lineage_order:
    s = markers[ln]['specificity']
    print(f"  {ln}: min={s.min():.3f} max={s.max():.3f} median={s.median():.3f}")

# ============================================================
# 4. 绘图
# ============================================================
print("\n" + "="*60 + "\nStep 4: 绘图\n" + "="*60)

def umap_arrows(ax):
    x0,y0,Larr = -0.02,-0.02,0.15
    ax.annotate('', xy=(x0+Larr,y0), xytext=(x0,y0), xycoords='axes fraction',
                arrowprops=dict(arrowstyle="->", color="black", lw=1.2))
    ax.text(x0+Larr/2, y0-0.02, 'UMAP1', transform=ax.transAxes, ha='center',
            va='top', fontsize=10, fontweight='bold')
    ax.annotate('', xy=(x0,y0+Larr), xytext=(x0,y0), xycoords='axes fraction',
                arrowprops=dict(arrowstyle="->", color="black", lw=1.2))
    ax.text(x0-0.02, y0+Larr/2, 'UMAP2', transform=ax.transAxes, ha='right',
            va='center', rotation=90, fontsize=10, fontweight='bold')

# ---- Panel a: 细胞类型 UMAP ----
fig = plt.figure(figsize=(10,5)); fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(1, L, figure=fig, wspace=0.20)
for idx,(traj,d) in enumerate(all_data.items()):
    ax = fig.add_subplot(gs[0,idx])
    if idx==0: add_label(ax,'a')
    adata=d['adata']; umap=adata.obsm['X_umap']; cats=d['cfg']['cell_order']
    cmap=plt.cm.Set1(np.linspace(0,0.8,len(cats)))
    for i,ct in enumerate(cats):
        m=adata.obs['cell_type']==ct
        if m.sum()>0:
            ax.scatter(umap[m,0],umap[m,1],s=4,c=[cmap[i]],alpha=0.6,label=ct,rasterized=True)
    ax.legend(loc='upper center',bbox_to_anchor=(0.5,-0.05),ncol=2,
              fontsize=14,markerscale=3,frameon=False)
    ax.set_title(traj,fontsize=16,color=d['cfg']['color'],fontweight='bold')
    ax.set_xticks([]);ax.set_yticks([])
    ax.spines[['top','right','bottom','left']].set_visible(False); umap_arrows(ax)
plt.tight_layout()
plt.savefig(output_dir+"Case1_a_celltype_umap.png",dpi=300,bbox_inches='tight')
plt.savefig(output_dir+"Case1_a_celltype_umap.pdf",bbox_inches='tight'); plt.close()
print("[a] done")

# ---- Panel b: 伪时序 UMAP ----
fig = plt.figure(figsize=(10,5)); fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(1, L, figure=fig, wspace=0.20)
for idx,(traj,d) in enumerate(all_data.items()):
    ax=fig.add_subplot(gs[0,idx])
    if idx==0: add_label(ax,'b')
    adata=d['adata']; umap=adata.obsm['X_umap']
    sc=ax.scatter(umap[:,0],umap[:,1],c=adata.obs['pseudotime'],
                  cmap='RdYlBu_r',s=4,alpha=0.6,rasterized=True)
    plt.colorbar(sc,ax=ax,shrink=0.6,pad=0.02)
    ax.set_title(traj,fontsize=16,color=d['cfg']['color'])
    ax.set_xticks([]);ax.set_yticks([])
    ax.spines[['top','right','bottom','left']].set_visible(False); umap_arrows(ax)
plt.tight_layout()
plt.savefig(output_dir+"Case1_b_pseudotime_umap.png",dpi=300,bbox_inches='tight')
plt.savefig(output_dir+"Case1_b_pseudotime_umap.pdf",bbox_inches='tight'); plt.close()
print("[b] done")

# ---- Panel c: 动态热图 ----
def rows_for_c(traj):
    gold=get_gold(traj); df=markers[traj].reset_index(drop=True)
    picked,seen=[],set()
    for tf in df['TF']:
        if tf not in seen: picked.append(tf); seen.add(tf)
        if len(picked)>=C_MAX_ROWS[traj]: break
    for tf in df['TF']:
        if is_validated(tf,gold) and tf not in seen: picked.append(tf); seen.add(tf)
    return picked

rows,labels,rcolors,rbold=[],[],[],[]
for traj in lineage_order:
    d=all_data[traj]; gold=get_gold(traj); color=d['cfg']['color']
    for tf in rows_for_c(traj):
        cur=safe_curve(d['tf_activity'],tf,'activity_raw')
        if cur is None: continue
        cur=uniform_filter1d(cur,size=5); sd=cur.std()
        rows.append((cur-cur.mean())/(sd if sd>1e-8 else 1.0))
        v=is_validated(tf,gold)
        labels.append(f"{'★ ' if v else '   '}{short(tf)} ({traj})")
        rcolors.append(color); rbold.append(v)
mat=np.array(rows); nr=len(rows)
fig=plt.figure(figsize=(9,max(6,0.46*nr))); fig.patch.set_facecolor('white')
ax=fig.add_subplot()
im=ax.imshow(mat,aspect='auto',cmap='RdBu_r',vmin=-2,vmax=2)
ax.set_yticks(range(nr)); ax.set_yticklabels(labels,fontsize=11)
for tk,b in zip(ax.get_yticklabels(),rbold): tk.set_fontweight('bold' if b else 'normal')
xspan=mat.shape[1]
for i,c in enumerate(rcolors):
    ax.add_patch(plt.Rectangle((-0.022*xspan-0.6,i-0.5),0.015*xspan,1.0,
                               color=c,clip_on=False,transform=ax.transData))
ax.set_xlabel('Pseudotime',fontsize=13); ax.set_xticks([]); add_label(ax,'c')
cb=plt.colorbar(im,ax=ax,shrink=0.5,pad=0.02); cb.set_label('Activity (z-score)',fontsize=11)
ax.set_title('Dynamic activity of top lineage-specific TFs\n'
             '(rows = unbiased predictions;  ★ / bold = literature-validated)',fontsize=12.5)
plt.tight_layout()
plt.savefig(output_dir+"Case1_c_dynamic_heatmap.png",dpi=300,bbox_inches='tight')
plt.savefig(output_dir+"Case1_c_dynamic_heatmap.pdf",bbox_inches='tight'); plt.close()
print(f"[c] done ({nr} rows)")

# ---- Panel d: turnover ----
fig,axes=plt.subplots(1,L,figsize=(2.8*L,3)); fig.patch.set_facecolor('white')
for idx,(traj,d) in enumerate(all_data.items()):
    ax=axes[idx]; turn=d['turnover']['smooth']
    ax.plot(time_axis,turn,lw=2.5,color=d['cfg']['color'])
    ax.fill_between(time_axis,0,turn,color=d['cfg']['color'],alpha=0.15)
    ax.set_title(traj,fontsize=16,fontweight='bold',color=d['cfg']['color'])
    ax.set_xlim(0,1); ax.set_ylim(0,None); ax.set_xticks([0,0.5,1]); ax.set_xticklabels([])
    ax.spines[['top','right']].set_visible(False)
    if idx==0:
        ax.set_ylabel('Edge Turnover\n(Jaccard distance)',fontsize=14); add_label(ax,'d')
fig.text(0.5,-0.02,'Pseudotime',ha='center',fontsize=16)
plt.tight_layout()
plt.savefig(output_dir+"Case1_d_turnover.png",dpi=300,bbox_inches='tight')
plt.savefig(output_dir+"Case1_d_turnover.pdf",bbox_inches='tight'); plt.close()
print("[d] done")

# ---- Panel e: 富集散点 (基线 0.5) ----
fig,axes=plt.subplots(1,L,figsize=(4.7*L,6.4),sharex=True); fig.patch.set_facecolor('white')
if L==1: axes=[axes]
xlo,xhi=E_XLIM
for ax,traj in zip(axes,lineage_order):
    gold=get_gold(traj); color=all_data[traj]['cfg']['color']
    df=markers[traj].reset_index(drop=True); spec=df['specificity'].values; N=len(df)
    y=np.arange(N)[::-1]
    val=np.array([is_validated(tf,gold) for tf in df['TF']])
    ax.scatter(spec[~val],y[~val],s=18,color='#d2d2d2',zorder=2,edgecolor='none')
    ax.scatter(spec[val],y[val],s=210,color=color,edgecolor='black',
               linewidth=1.1,marker='*',zorder=4)
    last=-1e9
    for i in sorted(np.where(val)[0],key=lambda k:-y[k]):
        ty=y[i]
        if ty-last < N*0.045: ty=last-N*0.045
        ax.annotate(short(df['TF'].iloc[i],14),(spec[i],y[i]),
                    xytext=(xhi-0.01,ty),textcoords='data',ha='right',va='center',
                    fontsize=8.5,fontweight='bold',color=color,
                    arrowprops=dict(arrowstyle='-',color=color,lw=0.6,alpha=0.5))
        last=ty
    v=validation[traj]; fold=v['fold']
    ax.axvline(BASELINE,color='gray',ls='--',lw=0.9,zorder=1)
    ax.text(BASELINE,N*1.02,'0.5\nbaseline',fontsize=7,color='gray',ha='center',va='bottom')
    ax.set_xlim(xlo,xhi); ax.set_ylim(-2,N*1.08); ax.set_yticks([])
    sub=f"{v['k']}/{v['K']} validated · {fold:.0f}× enriched · {pstr(v['pval'])}" if v['k']>0 \
        else f"{v['k']}/{v['K']} validated · {pstr(v['pval'])}"
    ax.set_title(f"{traj}\n{sub}",fontsize=10.5,fontweight='bold',color=color)
    ax.set_xlabel('Lineage specificity (max activity fraction)',fontsize=9)
    ax.text(xhi-0.01,-1.2,f"{N} candidate TFs",fontsize=8,color='#999',
            ha='right',va='top',style='italic')
    ax.spines[['top','right','left']].set_visible(False)
add_label(axes[0],'e')
axes[-1].legend(handles=[
    Line2D([0],[0],marker='*',color='w',markerfacecolor='#777',markeredgecolor='k',
           markersize=15,label='Validated (literature)'),
    Line2D([0],[0],marker='o',color='w',markerfacecolor='#d2d2d2',markersize=8,
           label='Predicted (novel)')],fontsize=8,frameon=False,loc='center right')
fig.text(0.5,-0.04,'Note: hematopoietic TFs are broadly pleiotropic; specificity is bounded '
         'below by 0.5 (two lineages),\nyet validated factors consistently rank at the top.',
         ha='center',fontsize=8,style='italic',color='#555')
plt.tight_layout()
plt.savefig(output_dir+"Case1_e_discovery.png",dpi=300,bbox_inches='tight')
plt.savefig(output_dir+"Case1_e_discovery.pdf",bbox_inches='tight'); plt.close()
print("[e] done")

# ---- Panel f: 共享 vs 特异靶基因 (双谱系: 交集 + 各自特异) ----
fig=plt.figure(figsize=(5,5)); fig.patch.set_facecolor('white')
ax_f=fig.add_subplot(); add_label(ax_f,'f')
gene_sets={}
for traj,d in all_data.items():
    genes=set(d['tf_gene_avg']['Gene'].unique())
    gene_sets[traj]=genes-{g for g in genes if str(g).startswith('chr')}
tn=list(gene_sets.keys()); s1,s2=[gene_sets[t] for t in tn]
cats={'Shared':len(s1&s2),f'{tn[0]}\nonly':len(s1-s2),f'{tn[1]}\nonly':len(s2-s1)}
bcolors=['#999999',all_data[tn[0]]['cfg']['color'],all_data[tn[1]]['cfg']['color']]
bars=ax_f.bar(range(len(cats)),list(cats.values()),color=bcolors,
              edgecolor='white',linewidth=1.2,alpha=0.9,width=0.66)
ax_f.set_xticks(range(len(cats))); ax_f.set_xticklabels(list(cats.keys()),fontsize=11)
for bar,val in zip(bars,cats.values()):
    ax_f.text(bar.get_x()+bar.get_width()/2,bar.get_height()+10,str(val),
              ha='center',fontsize=12,fontweight='bold')
ax_f.set_ylabel('Number of Target Genes',fontsize=12)
ax_f.set_title('Shared vs Lineage-Specific\nTarget Genes',fontsize=13)
ax_f.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(output_dir+"Case1_f_shared_genes.png",dpi=300,bbox_inches='tight')
plt.savefig(output_dir+"Case1_f_shared_genes.pdf",bbox_inches='tight'); plt.close()
print("[f] done")

# ---- Panel g: 谱系曲线 ----
nrow=L; ncol=max(G_N_COL.values())
fig,axes=plt.subplots(nrow,ncol,figsize=(2.15*ncol,2.2*nrow),sharey='row')
fig.patch.set_facecolor('white')
if nrow==1: axes=axes.reshape(1,-1)
for r,traj in enumerate(lineage_order):
    d=all_data[traj]; gold=get_gold(traj); color=d['cfg']['color']
    tfs=list(markers[traj]['TF'].head(G_N_COL[traj]))
    for h in [tf for tf in markers[traj]['TF'] if is_validated(tf,gold)]:
        if h not in tfs:
            for j in range(len(tfs)-1,-1,-1):
                if not is_validated(tfs[j],gold): tfs[j]=h; break
    for c in range(ncol):
        ax=axes[r,c]
        if c<len(tfs):
            tf=tfs[c]; cur=safe_curve(d['tf_activity'],tf,'activity_minmax')
            if cur is not None:
                cur=uniform_filter1d(cur,size=5)
                ax.plot(time_axis,cur,color=color,lw=2)
                ax.fill_between(time_axis,0,cur,color=color,alpha=0.12)
            v=is_validated(tf,gold)
            ax.set_title(f"{'★ ' if v else ''}{short(tf,13)}",fontsize=9.5,
                         color=color,fontweight='bold' if v else 'normal')
        else: ax.axis('off')
        ax.set_xlim(0,1); ax.set_xticks([]); ax.set_yticks([])
        ax.spines[['top','right']].set_visible(False)
    axes[r,0].set_ylabel(f'{traj}\nActivity',color=color,fontsize=10.5)
fig.text(0.5,-0.01,'Pseudotime',ha='center',fontsize=12)
fig.text(0.995,1.008,'★ / bold = literature-validated;  others = novel predictions',
         ha='right',fontsize=8.5,style='italic')
add_label(axes[0,0],'g')
plt.tight_layout()
plt.savefig(output_dir+"Case1_g_lineage_curves.png",dpi=300,bbox_inches='tight')
plt.savefig(output_dir+"Case1_g_lineage_curves.pdf",bbox_inches='tight'); plt.close()
print("[g] done")

# ============================================================
# 5. 补充面板
# ============================================================
fig,axes=plt.subplots(1,L,figsize=(2.8*L,3.5),sharey=True); fig.patch.set_facecolor('white')
for idx,(traj,d) in enumerate(all_data.items()):
    ax=axes[idx]; sc=d['region_gene']['avg_ts_weight'].values
    ax.hist(sc,bins=50,density=True,color=d['cfg']['color'],alpha=0.75,edgecolor='white',linewidth=0.5)
    mv=np.mean(sc); ax.axvline(mv,color='black',ls='--',lw=1.2,alpha=0.7)
    ax.text(mv,ax.get_ylim()[1]*0.85,f'$\\mu$={mv:.3f}',fontsize=12,va='top')
    ax.set_title(f'{traj}\n(n={len(sc):,})',fontsize=15,fontweight='bold',color=d['cfg']['color'])
    ax.spines[['top','right']].set_visible(False)
    if idx==0: ax.set_ylabel('Density',fontsize=14)
fig.text(0.5,-0.02,'Edge Probability',ha='center',fontsize=16)
plt.tight_layout(); plt.savefig(output_dir+"Supp_C_edge_prob.png",dpi=300,bbox_inches='tight'); plt.close()

fig,axes=plt.subplots(1,L,figsize=(2.8*L,3.5),sharey=True); fig.patch.set_facecolor('white')
for idx,(traj,d) in enumerate(all_data.items()):
    ax=axes[idx]; vals=tf_outdegrees[traj]['n_targets'].values
    ax.hist(vals,bins=30,density=True,color=d['cfg']['color'],alpha=0.75,edgecolor='white',linewidth=0.5)
    mv=np.median(vals); ax.axvline(mv,color='black',ls='--',lw=1.2,alpha=0.7)
    ax.text(mv,ax.get_ylim()[1]*0.9,f' med={mv:.0f}',fontsize=12,va='top')
    ax.set_title(traj,fontsize=15,fontweight='bold',color=d['cfg']['color'])
    ax.spines[['top','right']].set_visible(False)
    if idx==0: ax.set_ylabel('Density',fontsize=14)
fig.text(0.5,-0.02,'TF Outdegree (# Target Genes)',ha='center',fontsize=16)
plt.tight_layout(); plt.savefig(output_dir+"Supp_G_outdegree.png",dpi=300,bbox_inches='tight'); plt.close()

# Supp_I: 已知 TF 跨双谱系相对活性
tf_lineage={}
for traj,d in all_data.items():
    for tf in d['cfg']['known_tfs']: tf_lineage.setdefault(tf,traj)
ordered=[]
for ln in lineage_order: ordered+=sorted([tf for tf,l in tf_lineage.items() if l==ln])
hm=pd.DataFrame(index=ordered)
for traj in all_data:
    od=tf_outdegrees[traj].set_index('TF')
    hm[traj]=od['sum_weight'].reindex(hm.index).fillna(0)
hm=hm[hm.sum(axis=1)>0]; ordered=list(hm.index)
hm_pct=(hm.div(hm.mean(axis=1),axis=0)-1.0)[lineage_order]
fig=plt.figure(figsize=(5,7)); fig.patch.set_facecolor('white'); ax=fig.add_subplot()
sns.heatmap(hm_pct,ax=ax,cmap='RdBu_r',center=0,vmin=-0.3,vmax=0.3,
            cbar_kws={'label':'Relative activity\n(deviation from mean)','shrink':0.6},linewidths=0.5)
ax.tick_params(axis='y',length=0,pad=18); ax.tick_params(axis='x',length=0)
for lab in ax.get_yticklabels(): lab.set_ha('right')
lc={t:all_data[t]['cfg']['color'] for t in all_data}; trans=ax.get_yaxis_transform()
for i,tf in enumerate(ordered):
    ax.add_patch(plt.Rectangle((-0.06,i+0.12),0.038,0.76,facecolor=lc[tf_lineage[tf]],
                 edgecolor='none',transform=trans,clip_on=False,zorder=5))
ax.set_xlim(-0.5,hm_pct.shape[1])
ax.legend(handles=[Patch(facecolor=lc[t],label=t) for t in all_data],
          title='TF lineage',loc='upper left',bbox_to_anchor=(1.30,1.0),
          fontsize=9,title_fontsize=9,frameon=False)
ax.set_title('Known Lineage TFs\nAcross Trajectories',fontsize=12); ax.set_xlabel('')
plt.tight_layout(); plt.savefig(output_dir+"Supp_I_known_heatmap.png",dpi=300,bbox_inches='tight'); plt.close()

print("\n"+"="*60)
print("双谱系版全部完成。主图: a,b,c,d,e,f,g  补充: Supp_C, Supp_G, Supp_I")
print(f"输出目录: {output_dir}")
print("="*60)

Step 1: 加载双谱系 GRN
  myeloid: dyn=239318665, cells=857
  erythroid: dyn=298590199, cells=992
双谱系 无偏发现 → 金标准验证  N=811
[myeloid] K=10, k=6/10, 富集=16.2x, p=2.90e-07, ★=['CEBPA', 'CEBPB', 'FOS::JUNB', 'FOS::JUND', 'FOSL1::JUNB', 'IRF4', 'IRF8']
[erythroid] K=7, k=3/7, 富集=11.6x, p=1.45e-03, ★=['GATA1::TAL1', 'GATA2']

specificity 实际范围 (用于 e 图 x 轴):
  myeloid: min=0.535 max=0.624 median=0.548
  erythroid: min=0.524 max=0.613 median=0.537

Step 4: 绘图


/tmp/ipykernel_69283/2324549131.py:250: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


[a] done


/tmp/ipykernel_69283/2324549131.py:268: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


[b] done
[c] done (29 rows)
[d] done
[e] done
[f] done
[g] done

双谱系版全部完成。主图: a,b,c,d,e,f,g  补充: Supp_C, Supp_G, Supp_I
输出目录: /home/wuyan/dygmamba_project/DRIMA/data/case1/analysis_final_2lineage/


# Function analysis

In [4]:

#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
DRIMA — lineage/disease-SPECIFIC functional enrichment, two routes
==================================================================
Goal: recover lineage- / disease-SPECIFIC functions cleanly, fixing the bug
where the shared-set negative control was MORE significant than the specific
sets.

Why the old result failed (diagnosis from your run):
  - background was only ~1900 "expressed" genes, and BOTH the specific and the
    shared sets were drawn from that same pool. With such a small background and
    a shared set (566) larger than the specific sets (309/315), over-representation
    testing mechanically favours the larger set -> shared control wins. Statistical
    artefact, not biology.
  - "A-has / B-lacks" edges are the LOWEST-confidence edges (high-confidence edges
    appear in both networks -> fall into shared). So the "specific" set was noise.

Two routes, both designed to still yield SPECIFIC functions:

  ROUTE A  corrected over-representation
    * background = Enrichr genome-wide default (background=None), NOT the 1900
      expressed genes -> fair comparison, removes the size artefact
    * "specific" = high-confidence in A AND (absent OR much weaker) in B
      (delta-weight specific, not naive set subtraction)

  ROUTE B  rank-based GSEA (prerank)   <-- most robust, recommended
    * statistic per gene = (max strength in A) - (max strength in B)
    * positive pole = A-specific functions, negative pole = B-specific functions
    * no hard set cut -> immune to set-size / degree bias

Run both, compare which gives clean, direction-correct signal.

Weight column auto-detected; prefers avg_total_weight (your data has
avg_ts_weight [saturated ~1] and avg_total_weight [has dynamic range]).

Deps: numpy pandas matplotlib gseapy>=1.0
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 12, "axes.titlesize": 12, "axes.labelsize": 12})

SIG = 0.05
TOPK = 400
DELTA_MIN = 0.0   # for route A: require A_strength - B_strength > DELTA_MIN
LIBS_HUMAN = ["GO_Biological_Process_2021", "KEGG_2021_Human",
              "MSigDB_Hallmark_2020", "WikiPathway_2021_Human"]
LIBS_MOUSE = ["GO_Biological_Process_2021", "KEGG_2019_Mouse",
              "MSigDB_Hallmark_2020", "WikiPathways_2019_Mouse"]


# ---------------------------------------------------------------------------
# data helpers
# ---------------------------------------------------------------------------
def _weight_col(df):
    for c in ["avg_total_weight", "total_weight", "avg_weight", "weight",
              "score", "importance", "avg_ts_weight"]:
        if c in df.columns:
            return c
    return None


def load_tf_gene(path):
    df = pd.read_pickle(os.path.join(path, "pred_tf_gene.pkl"))
    df = df[~df["Gene"].astype(str).str.startswith("chr")].copy()
    df["Gene"] = df["Gene"].astype(str)
    return df


def gene_strength(df):
    """Per-gene strength = max regulatory weight over its incoming edges."""
    wc = _weight_col(df)
    if wc is None:
        # no weight -> use degree (number of regulating TFs) as a fallback proxy
        return df.groupby("Gene").size().astype(float)
    return df.groupby("Gene")[wc].max()


# ---------------------------------------------------------------------------
# enrichment runners (robust to empty / server errors, with retry)
# ---------------------------------------------------------------------------
def _enrichr(gene_list, background, organism, libraries, name, retries=2):
    import gseapy as gp
    import time
    all_res = []
    for gs in libraries:
        ok = False
        for attempt in range(retries + 1):
            kwargs = dict(gene_list=list(gene_list), gene_sets=gs,
                          organism=organism, outdir=None, no_plot=True)
            if background is not None:
                kwargs["background"] = list(background)
            try:
                enr = gp.enrichr(**kwargs)
                r = enr.results.copy()
                if r is not None and len(r) > 0:
                    r["library"] = gs
                    all_res.append(r)
                ok = True
                break
            except Exception as e:
                if attempt < retries:
                    time.sleep(2)
                    continue
                print(f"  [GO] {name}/{gs} failed after retries: {e}")
        if not ok:
            continue
    return all_res


def run_ora(gene_list, name, organism, libraries, out_csv=None, background=None):
    """ROUTE A: over-representation. background=None -> Enrichr genome default."""
    try:
        import gseapy as gp  # noqa
    except ImportError:
        print(f"  [ORA] gseapy not installed; skip {name}")
        return None
    gene_list = [g for g in gene_list if isinstance(g, str) and not g.startswith("chr")]
    if len(gene_list) < 10:
        print(f"  [ORA] {name}: {len(gene_list)} genes (<10), skipped")
        return None
    all_res = _enrichr(gene_list, background, organism, libraries, name)
    if not all_res:
        print(f"  [ORA] {name}: NO results (n={len(gene_list)}); first genes {gene_list[:5]}")
        return None
    res = pd.concat(all_res, ignore_index=True)
    pcol = "Adjusted P-value" if "Adjusted P-value" in res.columns else "P-value"
    res["_padj"] = res[pcol]
    res = res.sort_values("_padj").reset_index(drop=True)
    n_sig = int((res["_padj"] < SIG).sum())
    print(f"  [ORA] {name}: {len(res)} terms, {n_sig} sig (top: "
          f"{res.iloc[0]['Term'][:46]}, P_adj={res.iloc[0]['_padj']:.2g})")
    if out_csv:
        res.to_csv(out_csv, index=False)
    return res


def run_prerank(rank_series, name, organism, libraries, out_csv=None):
    """
    ROUTE B: rank-based GSEA. rank_series: index=gene, value=A_strength - B_strength.
    Positive NES -> enriched at A pole; negative NES -> enriched at B pole.
    """
    try:
        import gseapy as gp
    except ImportError:
        print(f"  [GSEA] gseapy not installed; skip {name}")
        return None
    rnk = rank_series.dropna().sort_values(ascending=False)
    rnk = rnk[~rnk.index.astype(str).str.startswith("chr")]
    if len(rnk) < 50:
        print(f"  [GSEA] {name}: only {len(rnk)} ranked genes, skipped")
        return None
    rnk_df = pd.DataFrame({0: rnk.index.astype(str), 1: rnk.values})
    frames = []
    for gs in libraries:
        try:
            pre = gp.prerank(rnk=rnk_df, gene_sets=gs, organism=organism,
                             min_size=5, max_size=1000, permutation_num=1000,
                             outdir=None, seed=0, no_plot=True)
            r = pre.res2d.copy()
            r["library"] = gs
            frames.append(r)
        except Exception as e:
            print(f"  [GSEA] {name}/{gs} failed: {e}")
    if not frames:
        print(f"  [GSEA] {name}: no results")
        return None
    res = pd.concat(frames, ignore_index=True)
    # gseapy prerank columns: Term, NES, NOM p-val, FDR q-val ...
    fdr_col = "FDR q-val" if "FDR q-val" in res.columns else \
              ("FDR" if "FDR" in res.columns else None)
    nes_col = "NES" if "NES" in res.columns else None
    if fdr_col and nes_col:
        res[fdr_col] = pd.to_numeric(res[fdr_col], errors="coerce")
        res[nes_col] = pd.to_numeric(res[nes_col], errors="coerce")
        res = res.sort_values(fdr_col).reset_index(drop=True)
        n_sig = int((res[fdr_col] < 0.25).sum())  # GSEA convention FDR<0.25
        pos = res[res[nes_col] > 0].head(1)
        neg = res[res[nes_col] < 0].head(1)
        print(f"  [GSEA] {name}: {len(res)} sets, {n_sig} FDR<0.25 | "
              f"A-pole top: {pos['Term'].values[:1]} | B-pole top: {neg['Term'].values[:1]}")
    if out_csv:
        res.to_csv(out_csv, index=False)
    return res


# ---------------------------------------------------------------------------
# specific-set builders
# ---------------------------------------------------------------------------
def delta_specific(df_a, df_b, k=TOPK, delta_min=DELTA_MIN):
    """
    High-confidence A-specific genes: strong in A AND (absent or much weaker in B).
    Returns the gene set.
    """
    sa = gene_strength(df_a)
    sb = gene_strength(df_b)
    allg = sa.index.union(sb.index)
    sa = sa.reindex(allg).fillna(0.0)
    sb = sb.reindex(allg).fillna(0.0)
    # normalize each to 0..1 so the delta is comparable
    def _norm(s):
        rng = s.max() - s.min()
        return (s - s.min()) / rng if rng > 1e-9 else s * 0
    na, nb = _norm(sa), _norm(sb)
    delta = (na - nb)
    cand = delta[delta > delta_min].sort_values(ascending=False)
    return set(cand.head(k).index.astype(str))


def delta_rank(df_a, df_b):
    """Per-gene ranking statistic = normalized strength_A - strength_B (for GSEA)."""
    sa = gene_strength(df_a)
    sb = gene_strength(df_b)
    allg = sa.index.union(sb.index)
    sa = sa.reindex(allg).fillna(0.0)
    sb = sb.reindex(allg).fillna(0.0)
    def _norm(s):
        rng = s.max() - s.min()
        return (s - s.min()) / rng if rng > 1e-9 else s * 0
    return (_norm(sa) - _norm(sb))


# ---------------------------------------------------------------------------
# Case 1 — haematopoiesis
# ---------------------------------------------------------------------------
def case1(case1_root, out_dir):
    print("\n=== Case 1 haematopoiesis: specific functions, two routes ===")
    os.makedirs(out_dir, exist_ok=True)
    mye = load_tf_gene(os.path.join(case1_root, "myeloid/process/"))
    ery = load_tf_gene(os.path.join(case1_root, "erythroid/process/"))

    # ---- ROUTE A: corrected over-representation, genome-wide background ----
    print(" -- Route A (ORA, genome background, delta-specific sets) --")
    mye_spec = delta_specific(mye, ery)
    ery_spec = delta_specific(ery, mye)
    print(f"    delta-specific: myeloid={len(mye_spec)}, erythroid={len(ery_spec)}")
    run_ora(mye_spec, "A_myeloid_specific", "human", LIBS_HUMAN,
            os.path.join(out_dir, "A_GO_myeloid.csv"), background=None)
    run_ora(ery_spec, "A_erythroid_specific", "human", LIBS_HUMAN,
            os.path.join(out_dir, "A_GO_erythroid.csv"), background=None)

    # ---- ROUTE B: rank-based GSEA (myeloid vs erythroid) ----
    print(" -- Route B (prerank GSEA, myeloid[+] vs erythroid[-]) --")
    rank = delta_rank(mye, ery)
    run_prerank(rank, "B_mye_vs_ery", "human", LIBS_HUMAN,
                os.path.join(out_dir, "B_GSEA_mye_vs_ery.csv"))
    print("    -> positive NES = myeloid-specific; negative NES = erythroid-specific")




In [5]:
CASE1_ROOT = "/home/wuyan/dygmamba_project/NewRealPlan/Case1/process/"
OUT        = "/home/wuyan/dygmamba_project/DRIMA/data/case1/function/"
os.makedirs(OUT, exist_ok=True)

case1(CASE1_ROOT, os.path.join(OUT, "case1"))

print("\nDone. Compare the two routes:")
print("  - Route A CSVs (A_GO_*): do the specific sets now beat a clean control?")
print("Diagnostic: if Route A still shows weak specific signal but Route B's")
print("poles are direction-correct (myeloid->immune, erythroid->heme), report")
print("Route B. If neither is clean, the honest reading is that specificity is")
print("combinatorial/diffuse rather than program-level (report that).")


=== Case 1 haematopoiesis: specific functions, two routes ===
 -- Route A (ORA, genome background, delta-specific sets) --
    delta-specific: myeloid=400, erythroid=400
  [ORA] A_myeloid_specific: 3109 terms, 69 sig (top: TNF-alpha Signaling via NF-kB, P_adj=3e-08)
  [ORA] A_erythroid_specific: 2805 terms, 1 sig (top: heme Metabolism, P_adj=4.6e-09)
 -- Route B (prerank GSEA, myeloid[+] vs erythroid[-]) --
  [GSEA] B_mye_vs_ery: 2847 sets, 43 FDR<0.25 | A-pole top: ['TNF-alpha Signaling via NF-kB'] | B-pole top: ['heme Metabolism']
    -> positive NES = myeloid-specific; negative NES = erythroid-specific

Done. Compare the two routes:
  - Route A CSVs (A_GO_*): do the specific sets now beat a clean control?
Diagnostic: if Route A still shows weak specific signal but Route B's
poles are direction-correct (myeloid->immune, erythroid->heme), report
Route B. If neither is clean, the honest reading is that specificity is
combinatorial/diffuse rather than program-level (report that).


## analysis

In [6]:

import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 13, "axes.titlesize": 13, "axes.labelsize": 13,
    "xtick.labelsize": 11, "ytick.labelsize": 9, "figure.dpi": 100,
})
SIG = 0.05




TERM_ALIASES = {
    "RNA splicing, via transesterification reactions with bulged adenosine as nucleophile":
        "RNA splicing (spliceosomal)",
    "RNA splicing, via transesterification reactions":
        "RNA splicing (via transesterification)",
    "Processing Of Capped Intron-Containing Pre-mRNA":
        "Processing of capped intron-containing pre-mRNA",
    "regulation of phosphatidylinositol 3-kinase signaling":
        "regulation of PI3K signaling",
    "negative regulation of sphingolipid biosynthetic process":
        "neg. regulation of sphingolipid biosynthesis",
    "negative regulation of ceramide biosynthetic process":
        "neg. regulation of ceramide biosynthesis",
    "regulation of ceramide biosynthetic process":
        "regulation of ceramide biosynthesis",
    "regulation of hemoglobin biosynthetic process":
        "regulation of hemoglobin biosynthesis",
    "C-type lectin receptor signaling pathway":
        "C-type lectin receptor signaling",
}
# ---------------------------------------------------------------------------
def _read_ora(csv):
    if not os.path.exists(csv):
        print(f"  [warn] missing {csv}")
        return None
    df = pd.read_csv(csv)
    pcol = "Adjusted P-value" if "Adjusted P-value" in df.columns else "P-value"
    df["_padj"] = pd.to_numeric(df[pcol], errors="coerce")
    return df.sort_values("_padj").reset_index(drop=True)


def _clean_term(t, maxlen=100):
    """Strip GO/Reactome/WP id suffixes, apply alias, only truncate if absurdly long."""
    t = str(t)
    t = t.split(" (GO")[0]
    t = t.split(" R-HSA")[0].split(" R-MMU")[0].split(" WP")[0]
    t = t.strip()
    if t in TERM_ALIASES:
        return TERM_ALIASES[t]
    key = t.rstrip(", ")
    if key in TERM_ALIASES:
        return TERM_ALIASES[key]
    return t if len(t) <= maxlen else t[:maxlen - 1] + "…"


def _wrap(t, width=30):
    return "\n".join(textwrap.wrap(t, width=width)) if len(t) > width else t





def draw_ora_bar(ax, df, title, color, top_n=10, note=None, wrap_width=30):
    if df is None or len(df) == 0:
        ax.text(0.5, 0.5, f"{title}\n[no data]", ha="center", va="center")
        ax.axis("off"); return
    top = df.head(top_n).iloc[::-1]
    terms = [_wrap(_clean_term(t), wrap_width) for t in top["Term"]]
    vals = -np.log10(top["_padj"].clip(lower=1e-300))
    sig_mask = top["_padj"].values < SIG
    colors = [color if s else "#BBBBBB" for s in sig_mask]
    ax.barh(range(len(terms)), vals, color=colors, alpha=0.9, edgecolor="white")
    ax.set_yticks(range(len(terms)))
    ax.set_yticklabels(terms, fontsize=8.5)
    ax.set_xlabel(r"$-\log_{10}$ adjusted $P$")
    ax.set_title(title, fontsize=13, fontweight="bold", pad=14)
    ax.axvline(-np.log10(SIG), ls="--", c="grey", lw=1)
    ax.text(-np.log10(SIG), len(terms) - 0.3, " P=0.05", color="grey",
            fontsize=8, va="top")
    if note:
        ax.text(0.97, 0.04, note, transform=ax.transAxes, ha="right", va="bottom",
                fontsize=8.5, style="italic", color="#555555")
    ax.spines[["top", "right"]].set_visible(False)




# ---------------------------------------------------------------------------
def case1_main(c1_dir, out_dir):
    print("=== Case 1 Fig 3J/3K (ORA main) ===")
    os.makedirs(out_dir, exist_ok=True)
    mye = _read_ora(os.path.join(c1_dir, "A_GO_myeloid.csv"))
    ery = _read_ora(os.path.join(c1_dir, "A_GO_erythroid.csv"))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    draw_ora_bar(axes[0], mye, "Fig 3J  Myeloid-specific targets", "#E41A1C",
                 note="immune / NF-κB / cytokine")
    draw_ora_bar(axes[1], ery, "Fig 3K  Erythroid-specific targets", "#FF7F00",
                 note="heme metabolism")
    fig.subplots_adjust(left=0.32, right=0.97, wspace=0.95, top=0.88, bottom=0.13)
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(out_dir, f"Fig3JK_lineage_GO.{ext}"),
                    dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved Fig3JK_lineage_GO.png/pdf -> {out_dir}")
    
    
    

def case1_gsea(c1_dir, out_dir, top_n=8, fdr_cut=0.25):
    print("=== Case 1 GSEA two-pole (supplementary) ===")
    os.makedirs(out_dir, exist_ok=True)
    csv = os.path.join(c1_dir, "B_GSEA_mye_vs_ery.csv")
    if not os.path.exists(csv):
        print(f"  [warn] missing {csv}"); return
    df = pd.read_csv(csv)
    nes = "NES"; fdr = "FDR q-val" if "FDR q-val" in df.columns else "FDR"
    df[nes] = pd.to_numeric(df[nes], errors="coerce")
    df[fdr] = pd.to_numeric(df[fdr], errors="coerce")
    df = df.dropna(subset=[nes, fdr])
    pos = df[(df[nes] > 0) & (df[fdr] < fdr_cut)].sort_values(fdr).head(top_n)
    neg = df[(df[nes] < 0) & (df[fdr] < fdr_cut)].sort_values(fdr).head(top_n)
    block = pd.concat([neg.iloc[::-1], pos.iloc[::-1]], ignore_index=True)
    if len(block) == 0:
        print("  [warn] no GSEA terms below FDR cut"); return
    terms = [_wrap(_clean_term(t), 28) for t in block["Term"]]
    vals = block[nes].values
    colors = ["#FF7F00" if v < 0 else "#E41A1C" for v in vals]
    fig, ax = plt.subplots(figsize=(9, max(4.5, 0.5 * len(terms) + 1.5)))
    ax.barh(range(len(terms)), vals, color=colors, alpha=0.9, edgecolor="white")
    ax.set_yticks(range(len(terms))); ax.set_yticklabels(terms, fontsize=9)
    ax.axvline(0, c="black", lw=0.8)
    ax.set_xlabel("GSEA NES   (myeloid →  |  ← erythroid)")
    ax.set_title("Myeloid-vs-erythroid GSEA (FDR < 0.25)", fontsize=13,
                 fontweight="bold", pad=12)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color="#E41A1C", label="Myeloid pole (+NES)"),
                       Patch(color="#FF7F00", label="Erythroid pole (−NES)")],
              fontsize=9, loc="lower right")
    ax.spines[["top", "right"]].set_visible(False)
    fig.subplots_adjust(left=0.40, right=0.96, top=0.92, bottom=0.12)
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(out_dir, f"SuppFig_GSEA_mye_ery.{ext}"),
                    dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved SuppFig_GSEA_mye_ery.png/pdf -> {out_dir}")

In [7]:
BASE_V2  = "/home/wuyan/dygmamba_project/DRIMA/data/case1/function/"
C1_DIR   = os.path.join(BASE_V2, "case1")
OUT      = "/home/wuyan/dygmamba_project/DRIMA/data/case1/function/"
case1_main(C1_DIR, OUT)

=== Case 1 Fig 3J/3K (ORA main) ===
  saved Fig3JK_lineage_GO.png/pdf -> /home/wuyan/dygmamba_project/DRIMA/data/case1/function/
